<h1 style="text-align: center;">Generador Modelo Prepago CMR</h1>


In [2]:
# Python Version : 3.12.4
import pandas as pd
import numpy as np
import os
import pyodbc

## 1. Leemos las bases de RF_BD_Gestion_RM

In [4]:
#Traemos los balances desde el Access de Liquidez

path = r"Z:\RF_PROCESOS\RF_Carteras\INTERFAZ_DATOS\RF_Base_Carteras_Completa.accdb"

db_driver = '{Microsoft Access Driver (*.mdb, *.accdb)}'
db_path = path
conn_str = (rf'DRIVER={db_driver};'
            rf'DBQ={db_path};')

conn = pyodbc.connect(conn_str)

df = (
    pd.read_sql(
        sql=f"""
            SELECT  RF_BD_Gestion_RM.Fec_Pro,
                    RF_BD_Gestion_RM.Fec_Vcto,
                    RF_BD_Gestion_RM.Cod_Sub_Pro,
                    RF_BD_Gestion_RM.Cap_Amort,
                    RF_BD_Gestion_RM.Int_Total_Cont
                    
            FROM RF_BD_Gestion_RM

            WHERE Cod_Pro = 'TARJETA DE CREDITO';

        """,
        con=conn
    )
).copy()

conn.close()

df.head()

C:\Users\fzentenos\AppData\Local\Temp\ipykernel_24352\2572361733.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql(


,Fec_Pro,Fec_Vcto,Cod_Sub_Pro,Cap_Amort,Int_Total_Cont
0,2025-08-19,2025-08-20,AVANCE,4.158330e+05,7.882500e+04
1,2025-08-19,2025-08-25,AVANCE,1.079984e+06,3.138080e+05
2,2025-08-19,2025-08-30,AVANCE,2.516627e+06,5.016560e+05
3,2025-08-19,2025-09-05,AVANCE,8.671190e+06,1.599396e+06
4,2025-08-19,2025-09-10,AVANCE,5.005411e+09,1.054788e+09


## 2. Cambiamos los nombres de las columnas

In [6]:
df.rename(columns={
    'Fec_Pro': 'FECHA_PROCESO',
    'Fec_Vcto': 'FECHA_VENCIMIENTO_CUOTA',
    'Cod_Sub_Pro': 'CODIGO_SUBPRODUCTO',
    'Cap_Amort': 'AMORTIZACION',
    'Int_Total_Cont': 'INTERES'
}, inplace=True)

df.head()

,FECHA_PROCESO,FECHA_VENCIMIENTO_CUOTA,CODIGO_SUBPRODUCTO,AMORTIZACION,INTERES
0,2025-08-19,2025-08-20,AVANCE,4.158330e+05,7.882500e+04
1,2025-08-19,2025-08-25,AVANCE,1.079984e+06,3.138080e+05
2,2025-08-19,2025-08-30,AVANCE,2.516627e+06,5.016560e+05
3,2025-08-19,2025-09-05,AVANCE,8.671190e+06,1.599396e+06
4,2025-08-19,2025-09-10,AVANCE,5.005411e+09,1.054788e+09


## 3. Agregamos Columnas y Separamos por SAV y NO SAV

In [8]:
# Extraemos el dia de fecha vencimiento para aplicar en modelo
df['Dia_F'] = df['FECHA_VENCIMIENTO_CUOTA'].dt.day
df['Dia_F'] = df['Dia_F'].replace({28: 30, 29: 30})

# Calculamos diferencia para filtrar en modelo
diff = (df['FECHA_VENCIMIENTO_CUOTA'] - df['FECHA_PROCESO']).dt.days
df['Plazo_Antiguo'] = diff.apply(lambda x: 'NO' if x > -1 else 'SI')

#Filtramos las columnas que utilizaremos en el modelo
clean_df = df[["FECHA_PROCESO", "FECHA_VENCIMIENTO_CUOTA", "Dia_F", "Plazo_Antiguo", "CODIGO_SUBPRODUCTO", "AMORTIZACION", "INTERES"]]

## filtramos los vcdos con los vigentes:
df_vcdos_mora = clean_df[clean_df['CODIGO_SUBPRODUCTO'].str.contains("MORA", case=False, na=False)]
df_vcdos_mora["Plazo_Antiguo"] = "NO"

clean_df = clean_df[~clean_df['CODIGO_SUBPRODUCTO'].str.contains("MORA", case=False, na=False)]

#Separamos productos SAV y NO SAV
df_sav = clean_df[clean_df['CODIGO_SUBPRODUCTO'].str.startswith('SUPER AVANCE')]
df_NO_sav = clean_df[~(clean_df['CODIGO_SUBPRODUCTO'].str.startswith('SUPER AVANCE'))]
#Agregamos los vencidos a No_SAV
df_NO_sav = pd.concat([df_vcdos_mora, df_NO_sav], ignore_index=True)

C:\Users\fzentenos\AppData\Local\Temp\ipykernel_24352\4135520988.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_vcdos_mora["Plazo_Antiguo"] = "NO"


## 4. Creamos funcion para calcular flujos con matriz

In [11]:
#Aca debemos crear la funcion para darle los inputs
def generador_matriz_SAV(df_sav, dia_f, plazo_antiguo, SMM_Porcentaje = 0.7866, Factor = 1):
    filtered_df = df_sav[
        (df_sav['Dia_F'] == dia_f) &
        (df_sav['Plazo_Antiguo'] == plazo_antiguo)] #NO
    
    grouped_df = (filtered_df.groupby('FECHA_VENCIMIENTO_CUOTA', as_index=False)[['AMORTIZACION', "INTERES"]].sum())
    
    grouped_df = grouped_df.sort_values('FECHA_VENCIMIENTO_CUOTA')
    flujos_matriz = grouped_df.to_numpy()
    
    #Creamos el vector de Plazo_SMM
    SMM = SMM_Porcentaje/100
    SMM_Vector = np.full((90), SMM) 
    Comp_SMM = np.zeros(90)
    
    for i in range(Comp_SMM.shape[0]):
        if i == 0:
            Comp_SMM[i] = (1-Factor * SMM_Vector[i])
        else:
            Comp_SMM[i] = Comp_SMM[i-1] * (1-Factor * SMM_Vector[i])
    
    
    # Aplicamos la matriz
    N = int(len(flujos_matriz))
    M = 90
    matriz = np.zeros((N, M))
    
    # Cambianos el recorrido de la matriz para que pudiese sumar las columnas antes
    for j in range(M):
        for i in range(N):
            if i == 0 and j == 0:
                matriz[i, j] = flujos_matriz[i,1] + flujos_matriz[i,2] + flujos_matriz[1:,1].sum()*SMM_Vector[j]*Factor
            elif i != 0 and j == 0:
                matriz[i, j] = flujos_matriz[i,1] * (1 - SMM_Vector[j] * Factor)
            elif i == j : #<= N
                 matriz[i, j] = matriz[i, j-1] + flujos_matriz[i,2]*Comp_SMM[j-1]+matriz[i+1:, j-1].sum()*SMM_Vector[j]*Factor
            elif i > j:
                matriz[i,j] = matriz[i, j-1] * (1-SMM_Vector[j]*Factor)
            else:
                matriz[i, j] = 0
    
    # calculamos las ultimas columnas despues de la matriz
    Vector_CW = SMM_Vector*Factor
    Vector_CX = np.zeros(N)
    for i in range(Vector_CX.shape[0]):
        if i == 0:
            Vector_CX[i] = 1
        else:
            Vector_CX[i] = Vector_CX[i-1]*(1-Vector_CW[i-1])
    
    #Calculamos la columna Prepago
    Vector_Prepago = np.zeros(N)
    for j in range(M):
        for i in range(N):
            if j==i:
                Vector_Prepago[i] = matriz[i,j]
    
    # IntereCes y Capital Prepago
    Int_CapPre = np.zeros((N, 2))
    for i in range(N):
        Int_CapPre[i,0] = max(Vector_CX[i] * flujos_matriz[i,2], 0)       
        Int_CapPre[i,1] = max(Vector_Prepago[i] - Int_CapPre[i,0], 0)
    
    #Creamos el vector para ser pegado en la hoja desarrollo
    fecha_venc = flujos_matriz[:, 0]
    amortizacion = Int_CapPre[:, 1]
    interes = Int_CapPre[:, 0]
    
    if SMM_Porcentaje == 0:
        cod_sub_pro_extension = "_NO_SAV_"
    else:
        cod_sub_pro_extension = "_SAV_"

    if Factor == 1.0:
        cod_pro = "MT_R13_TC_CMR_BASE"
        cod_sub_pro = "MT_R13_TC_CMR"+cod_sub_pro_extension+"BASE"
    elif Factor == 0.8:
        cod_pro = "MT_R13_TC_CMR_UP"
        cod_sub_pro = "MT_R13_TC_CMR"+cod_sub_pro_extension+"UP"
    elif Factor == 1.2:
        cod_pro = "MT_R13_TC_CMR_DOWN"
        cod_sub_pro = "MT_R13_TC_CMR"+cod_sub_pro_extension+"DOWN"

    #Creamos la hoja desarrollo
    df_desarrollo = pd.DataFrame({
        "FECHA PROCESO": np.full((N), clean_df["FECHA_PROCESO"].iloc[0]) ,
        "CODIGO_EMPRESA": np.full((N), 3),
        "OPERACION": np.full((N), np.nan),  
        "COD ACT/PAS": np.full((N), "ACT"),
        "MONEDA_ORIGEN": np.full((N), "CLP"),
        "MONEDA_COMPENSACION": np.full((N), "CLP"),
        "COMPENSACION": np.full((N), np.nan),  
        "CODIGO_PRODUCTO": np.full((N), cod_pro),
        "CODIGO_SUBPRODUCTO": np.full((N), cod_sub_pro),
        "FECHA CREACION": np.full((N), np.nan),  
        "NUMERO_CUOTA": np.full((N), np.nan),
        "FECHA_INICIO_CUOTA": np.full((N), np.nan),
        "FECHA_VENCIMIENTO_CUOTA": fecha_venc,
        "FECHA PAGO": fecha_venc,  # vacío
        "FECHA_REPRICING": fecha_venc,  # vacío
        "AMORTIZACION": amortizacion,
        "INTERES": interes,
        "INTERES_DEVENGADO": np.full((N), np.nan),  # vacío
        "VP_AMORTIZACION": np.full((N), np.nan),  # vacío
        "VP_INTERES": np.full((N), np.nan),  # vacío
        "FACTOR DE RIESGO": np.full((N), np.nan),  # vacío
        "TIPO_CUOTA": np.full((N), 1),
        "AREA NEGOCIO": np.full((N), "CMR_BKG"),
        "CODIGO_ EJECUTIVO": np.full((N), np.nan),  # vacío
        "CODIGO_ESTRATEGIA": np.full((N), "CMR_BKG"),
        "CLASIFICACION_CONTABLE": np.full((N), "HTM"),
        "TIPO TASA": np.full((N), 1),
        "INDEXADOR": np.full((N), np.nan),
        "TASA": np.full((N), np.nan),
        "TASA CF": np.full((N), np.nan),
        "SPREAD": np.full((N), np.nan),
    })
    
    df_desarrollo = pd.concat([df_desarrollo], ignore_index=True)
    return df_desarrollo


## 5. Corremos funcion para SAV y NO SAV

In [13]:
#definimos las fechas de facturacion y Factores
#Fechas_Facturacion = [5, 10, 15, 20, 25, 30]
Fechas_Facturacion = df.Dia_F.unique()
Factores = [1.0, 0.8, 1.2]
#Creamos el DF que se irá concatenando para cada caso
df_desarrollo = pd.DataFrame()

#Corremos para SAV
SMM_SAV = 0.7866
for Valor_Factor in Factores:
    for Fecha_Fac in Fechas_Facturacion:
        df_ciclo = generador_matriz_SAV(df_sav, Fecha_Fac, "NO", SMM_Porcentaje = SMM_SAV, Factor = Valor_Factor)
        df_desarrollo = pd.concat([df_desarrollo, df_ciclo], ignore_index=True)

#Corremos para NO SAV
SMM_NO_SAV = 0
for Valor_Factor in Factores:
    for Fecha_Fac in Fechas_Facturacion:
        df_ciclo = generador_matriz_SAV(df_NO_sav, Fecha_Fac, "NO", SMM_Porcentaje = SMM_NO_SAV, Factor = Valor_Factor)
        df_desarrollo = pd.concat([df_desarrollo, df_ciclo], ignore_index=True)

C:\Users\fzentenos\AppData\Local\Temp\ipykernel_24352\2358609241.py:13: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_desarrollo = pd.concat([df_desarrollo, df_ciclo], ignore_index=True)
C:\Users\fzentenos\AppData\Local\Temp\ipykernel_24352\2358609241.py:13: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_desarrollo = pd.concat([df_desarrollo, df_ciclo], ignore_index=True)
C:\Users\fzentenos\AppData\Local\Temp\ipykernel_24352\2358609241.py:13: FutureWarning: The behavior of DataFrame concate

## 6. Validamos los resultados del modelo vs los datos de la BD

In [15]:
# Agrupar por 'categoria' y sumar 'ventas' y 'costos'
df_validador = df_desarrollo.groupby('CODIGO_SUBPRODUCTO')[['AMORTIZACION', 'INTERES']].sum().reset_index()

#Hacer las sumas operaciones antes de esto, por el formato...
df_validador['AMORTIZACION'] = df_validador['AMORTIZACION'].apply(lambda x: '{:,.0f}'.format(x))
df_validador['INTERES'] = df_validador['INTERES'].apply(lambda x: '{:,.0f}'.format(x))
print(df_validador)
print()

capital_total_bd = '{:,.0f}'.format(df.AMORTIZACION.sum()).replace(',', 'X').replace('.', ',').replace('X', '.')
capital_total_modelo = '{:,.0f}'.format((df_desarrollo.AMORTIZACION.sum() / 3)).replace(',', 'X').replace('.', ',').replace('X', '.')

#Comparamos los montos traidos de la BD con los montos que arroja el modelo
print("Capitales: (BD vs Modelo)", capital_total_bd, "vs", capital_total_modelo)
if abs(df.AMORTIZACION.sum() - df_desarrollo.AMORTIZACION.sum() / 3) < 1:
    print("Capitales OK ✅")
else:
    print("Error en total de capitales ❌")

print("Fecha proceso:", df_desarrollo["FECHA PROCESO"].iloc[0].strftime('%d-%m-%Y'))

          CODIGO_SUBPRODUCTO       AMORTIZACION          INTERES
0  MT_R13_TC_CMR_NO_SAV_BASE  2,082,946,800,846  186,099,323,459
1  MT_R13_TC_CMR_NO_SAV_DOWN  2,082,946,800,846  186,099,323,459
2    MT_R13_TC_CMR_NO_SAV_UP  2,082,946,800,846  186,099,323,459
3     MT_R13_TC_CMR_SAV_BASE    775,955,404,623  211,854,053,385
4     MT_R13_TC_CMR_SAV_DOWN    775,955,404,623  208,142,648,337
5       MT_R13_TC_CMR_SAV_UP    775,955,404,623  215,683,266,864

Capitales: (BD vs Modelo) 2.858.902.205.469 vs 2.858.902.205.469
Capitales OK ✅
Fecha proceso: 19-08-2025


In [16]:
## Faltaria agregar un validador historico (ir a buscar los ultimos x modelos en la ruta y compararlos...)
# Podriamos agregar un validador de 30 y 90 dias similar al de TC-CMR

## 7. Exportamos el resultado del modelo

In [18]:
# Archivo Proceso Diario
fecha_proceso = clean_df["FECHA_PROCESO"].iloc[0].strftime('%Y%m%d')
ruta = r'Z:\RF_PROCESOS\RF_Modelos\RF_Modelo_Prepago_CMR'

# Nombre del archivo con ruta completa
nombre_archivo = os.path.join(ruta, 'Prepago_TC_CMR.xlsx')

with pd.ExcelWriter(nombre_archivo, engine='xlsxwriter', datetime_format='dd-mm-yyyy') as writer:
    df_desarrollo.to_excel(writer, index=False, sheet_name='DESARROLLO')
    workbook  = writer.book
    worksheet = writer.sheets['DESARROLLO']

    # Ajustar ancho de columnas automáticamente según contenido
    for i, col in enumerate(df_desarrollo.columns):
        # Obtener el ancho máximo entre el nombre de la columna y el contenido de esa columna
        max_len = max(
            df_desarrollo[col].astype(str).map(len).max(),  # largo máximo de datos
            len(col)  # largo del encabezado
        ) + 2  # margen extra para que no se vea apretado
        worksheet.set_column(i, i, max_len)


# Historia
ruta = r'Z:\RF_PROCESOS\RF_Modelos\RF_Modelo_Prepago_CMR\Prepago_CMR_Historia'
nombre_archivo = os.path.join(ruta, f'{fecha_proceso}_Prepago_TC_CMR.xlsx')

with pd.ExcelWriter(nombre_archivo, engine='xlsxwriter', datetime_format='dd-mm-yyyy') as writer:
    df_desarrollo.to_excel(writer, index=False, sheet_name='DESARROLLO')
    worksheet = writer.sheets['DESARROLLO']

    for i, col in enumerate(df_desarrollo.columns):
        max_len = max(df_desarrollo[col].astype(str).map(len).max(), len(col)) + 2
        worksheet.set_column(i, i, max_len)


## 8. Fin Proceso Prepago CMR

                    Equipo de modelos y metodologias, Gerencia de riesgo financiero del Banco Falabella Chile.